# Od chaosu plików do czystej ramki danych

## Obróbka danych — `pathlib`, `glob`, `re`, `pandas.str`

**Cel zajęć:** mamy katalog z pomiarami środowiskowymi z kilku stacji w Łodzi. Pliki są w różnych podkatalogach, mają różne formaty nazw, w środku różne separatory. Naszym zadaniem jest **wczytać wszystko, wyciągnąć metadane z nazw plików** (stacja, data, typ pomiaru) i **połączyć w jedną ramkę** gotową do analizy.

In [4]:
import zipfile
from pathlib import Path

zip_path = Path("dane_lodz.zip")
output_dir = Path("dane_lodz")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_dir)

## 1. Rozpoznanie terenu

Zacznijmy od obejrzenia, z czym mamy do czynienia.

In [5]:
from pathlib import Path

ROOT = Path("dane_lodz")

# Sprawdzenie, czy katalog istnieje
if ROOT.exists() and ROOT.is_dir():
    for p in ROOT.iterdir():
        print(p)
else:
    print(f"Katalog '{ROOT}' nie istnieje.")

dane_lodz/dane_lodz


In [6]:
# Pokaż drzewo - kilka pierwszych plików z każdego katalogu
for subdir in sorted(ROOT.iterdir()):
    if subdir.is_dir():
        print(f"\n📁 {subdir.name}/")
        for f in sorted(subdir.iterdir())[:3]:
            print(f"   {f.name}")
        print("   ...")


📁 dane_lodz/
   jakosc_powietrza
   notatki.txt
   temperatura
   ...


## 2. `pathlib` — podstawy

`pathlib` zastępuje `os.path`. Główna klasa: `Path`. Działa **cross-platform** — używaj `/` zamiast `os.path.join`.

In [7]:
# Operator / buduje ścieżki
plik = ROOT / "temperatura" / "temp_ST01_polesie_2025-03-01.csv"
print("Pełna ścieżka:    ", plik)
print("Nazwa pliku:      ", plik.name)
print("Sama nazwa:       ", plik.stem)
print("Rozszerzenie:     ", plik.suffix)
print("Katalog nadrzędny:", plik.parent)
print("Czy istnieje:     ", plik.exists())

Pełna ścieżka:     dane_lodz/temperatura/temp_ST01_polesie_2025-03-01.csv
Nazwa pliku:       temp_ST01_polesie_2025-03-01.csv
Sama nazwa:        temp_ST01_polesie_2025-03-01
Rozszerzenie:      .csv
Katalog nadrzędny: dane_lodz/temperatura
Czy istnieje:      False


### **Ćwiczenie 1**
Dla pliku `plik` powyżej wypisz:
- nazwę bez rozszerzenia
- nazwę katalogu nadrzędnego (samą nazwę, nie pełną ścieżkę)
- rozmiar pliku w bajtach (podpowiedź: `.stat().st_size`)

In [9]:
# Ćwiczenie 1 — rozwiązanie
print(f"Nazwa bez rozszerzenia: {plik.stem}")
print(f"Nazwa katalogu nadrzędnego: {plik.parent.name}")
print(f"Rozmiar pliku: {plik.stat().st_size} bajtów")

Nazwa bez rozszerzenia: temp_ST01_polesie_2025-03-01
Nazwa katalogu nadrzędnego: temperatura


FileNotFoundError: [Errno 2] No such file or directory: 'dane_lodz/temperatura/temp_ST01_polesie_2025-03-01.csv'

## 3. `glob` i `rglob` — znajdowanie plików

- `Path.glob(pattern)` — w danym katalogu (jeden poziom)
- `Path.rglob(pattern)` — **rekurencyjnie** (we wszystkich podkatalogach)
- wzorce: `*` (cokolwiek), `?` (jeden znak), `[abc]` (zbiór znaków)

In [10]:
# Wszystkie pliki CSV w całym drzewie
csv_files = list(ROOT.rglob("*.csv"))
print(f"Znaleziono plików CSV: {len(csv_files)}")
for f in csv_files[:5]:
    print(" ", f)

Znaleziono plików CSV: 80
  dane_lodz/dane_lodz/jakosc_powietrza/pm25-ST03_baluty-20250301.csv
  dane_lodz/dane_lodz/jakosc_powietrza/pm25-ST02_widzew-20250308.csv
  dane_lodz/dane_lodz/jakosc_powietrza/pm25-ST01_polesie-20250302.csv
  dane_lodz/dane_lodz/jakosc_powietrza/pm25-ST02_widzew-20250306.csv
  dane_lodz/dane_lodz/jakosc_powietrza/pm25-ST01_polesie-20250303.csv


In [11]:
# Tylko pliki z temperatury
temp_files = list((ROOT / "temperatura").glob("temp_*.csv"))
print(f"Plików temperatury: {len(temp_files)}")

Plików temperatury: 0


In [12]:
# UWAGA: glob łapie też śmieci - np. backupy. Trzeba filtrować.
podejrzane = list(ROOT.rglob("*backup*"))
print("Pliki które chcemy zignorować:")
for f in podejrzane:
    print(" ", f)

Pliki które chcemy zignorować:
  dane_lodz/dane_lodz/temperatura/_old_backup.csv.bak


### **Ćwiczenie 2**
Znajdź:
1. Wszystkie pliki `.txt` w drzewie
2. Wszystkie pliki PM2.5 (zaczynają się od `pm25-`)
3. Wszystkie pliki temperatury, **z wyjątkiem** wersji `_v2`

In [13]:
import re

# 1. Wszystkie pliki .txt
txt_files = list(ROOT.rglob("*.txt"))
print(f"1. Pliki .txt w drzewie: {len(txt_files)}")
for f in txt_files[:3]:
    print(f"   {f.name}")

# 2. Pliki PM2.5
pm25_files = list(ROOT.rglob("pm25-*.csv"))
print(f"\n2. Pliki PM2.5: {len(pm25_files)}")
for f in pm25_files[:3]:
    print(f"   {f.name}")

# 3. Pliki temperatury BEZ wersji _v2 (filtrowanie przez re)
temp_bez_v2 = [f for f in (ROOT/"temperatura").glob("temp_*.csv")
               if not re.search(r'_v\d', f.name)]
print(f"\n3. Pliki temperatury BEZ wersji _v2: {len(temp_bez_v2)}")
for f in sorted(temp_bez_v2)[:3]:
    print(f"   {f.name}")

1. Pliki .txt w drzewie: 42
   notatki.txt
   info.txt
   humidity_ST04_gorna_v2_2025_03_07.txt

2. Pliki PM2.5: 40
   pm25-ST03_baluty-20250301.csv
   pm25-ST02_widzew-20250308.csv
   pm25-ST01_polesie-20250302.csv

3. Pliki temperatury BEZ wersji _v2: 0


## 4. Wyrażenia regularne — krótkie powtórzenie

Regex to **język opisywania wzorców tekstu**. Najważniejsze elementy:

| Wzorzec | Znaczenie |
|---------|-----------|
| `\d` | cyfra (0-9) |
| `\w` | znak "słowny" (litera, cyfra, `_`) |
| `\s` | biały znak |
| `.` | dowolny znak |
| `+` | jeden lub więcej |
| `*` | zero lub więcej |
| `{n}` | dokładnie n |
| `{n,m}` | od n do m |
| `[abc]` | jeden ze znaków |
| `(...)` | grupa |
| `(?P<nazwa>...)` | **grupa nazwana** ← KLUCZOWE |

Funkcje z `re`:
- `re.search(pattern, text)` — szuka **gdziekolwiek** w tekście
- `re.match(pattern, text)` — szuka **od początku**
- `re.findall(pattern, text)` — wszystkie dopasowania
- `re.sub(pattern, replacement, text)` — zamiana

In [14]:
# Prosty przykład: wyciągnij datę z nazwy pliku
nazwa = "temp_ST01_polesie_2025-03-01.csv"
wzorzec = r"(\d{4}-\d{2}-\d{2})"
m = re.search(wzorzec, nazwa)
print(m.group(1))

2025-03-01


In [15]:
# Wzorzec dla plików temperatury:
# temp_<STACJA>_<DATA>.csv  lub  temp_<STACJA>_<DATA>_v2.csv
wzorzec_temp = re.compile(
    r"temp_(?P<stacja>ST\d{2}_\w+?)_(?P<data>\d{4}-\d{2}-\d{2})(?:_v(?P<wersja>\d+))?\.csv"
)

for f in list((ROOT / "temperatura").glob("temp_*.csv"))[:5]:
    m = wzorzec_temp.match(f.name)
    if m:
        print(f.name)
        print("  →", m.groupdict())

Rozbiór wzorca:
- `temp_` — literalny początek
- `(?P<stacja>ST\d{2}_\w+?)` — `ST` + 2 cyfry + `_` + nazwa (leniwa, żeby nie zjeść daty)
- `_` — separator
- `(?P<data>\d{4}-\d{2}-\d{2})` — data w formacie ISO
- `(?:_v(?P<wersja>\d+))?` — **opcjonalna** wersja, `(?:...)` to grupa nie-przechwytująca
- `\.csv` — kropka literalna (uciekamy `\.`) + rozszerzenie

### **Ćwiczenie 3**

Napisz regex dla plików PM2.5 (przykład: `pm25-ST01_polesie-20250301.csv`).
Wyciągnij stację i datę. Uwaga: data nie ma separatorów!

I drugi — dla wilgotności (np. `humidity_ST02_widzew_v1_2025_03_05.txt`).
Tu wersja jest **w środku**, nie na końcu.

In [16]:
# Ćwiczenie 3 — rozwiązanie

# PM2.5: pm25-ST01_polesie-20250301.csv
# data bez separatorów: YYYYMMDD
wzorzec_pm25 = re.compile(
    r"pm25-(?P<stacja>ST\d{2}_\w+)-(?P<data>\d{8})\.csv"
)

# Wilgotność: humidity_ST02_widzew_v1_2025_03_05.txt
# wersja w środku, data z podkreślnikami
wzorzec_hum = re.compile(
    r"humidity_(?P<stacja>ST\d{2}_\w+)_v(?P<wersja>\d+)_(?P<rok>\d{4})_(?P<miesiac>\d{2})_(?P<dzien>\d{2})\.txt"
)

print("=== Regex dla PM2.5 ===")
for f in list((ROOT/"jakosc_powietrza").glob("pm25-*.csv"))[:3]:
    m = wzorzec_pm25.match(f.name)
    if m:
        print(f.name)
        print("  →", m.groupdict())

print("\n=== Regex dla wilgotności ===")
for f in list((ROOT/"wilgotnosc").glob("humidity_*.txt"))[:3]:
    m = wzorzec_hum.match(f.name)
    if m:
        print(f.name)
        print("  →", m.groupdict())

=== Regex dla PM2.5 ===

=== Regex dla wilgotności ===


## 5. Łączymy: pathlib + regex + pandas

Teraz właściwa robota. Strategia:
1. Znajdź pliki danego typu (`rglob` + regex do walidacji nazwy)
2. Wyciągnij metadane z nazwy
3. Wczytaj do pandas
4. Dodaj kolumny z metadanymi
5. Złóż wszystko w jedną ramkę

In [17]:
import pandas as pd

def wczytaj_temperature(root: Path) -> pd.DataFrame:
    """Wczytuje wszystkie pliki temperatury, dodaje metadane, łączy w jedną ramkę."""
    ramki = []
    for f in root.rglob("temp_*.csv"):
        m = wzorzec_temp.match(f.name)
        if not m:
            print(f"⚠️  Pomijam: {f.name}")
            continue
        meta = m.groupdict()
        df = pd.read_csv(f)
        df["stacja"] = meta["stacja"]
        df["data_pliku"] = meta["data"]
        df["wersja"] = meta.get("wersja") or "1"
        df["plik"] = f.name
        ramki.append(df)
    return pd.concat(ramki, ignore_index=True)

temp_df = wczytaj_temperature(ROOT)
print("Kształt:", temp_df.shape)
temp_df.head()

Kształt: (320, 7)


,timestamp,temperatura_C,uwagi,stacja,data_pliku,wersja,plik
0,2025-03-01 00:00:00,8.45,ok,ST04_gorna,2025-03-01,1,temp_ST04_gorna_2025-03-01.csv
1,2025-03-01 03:00:00,4.85,ok,ST04_gorna,2025-03-01,1,temp_ST04_gorna_2025-03-01.csv
2,2025-03-01 06:00:00,5.91,kalibracja,ST04_gorna,2025-03-01,1,temp_ST04_gorna_2025-03-01.csv
3,2025-03-01 09:00:00,19.9 C,NaN,ST04_gorna,2025-03-01,1,temp_ST04_gorna_2025-03-01.csv
4,2025-03-01 12:00:00,14.11,NaN,ST04_gorna,2025-03-01,1,temp_ST04_gorna_2025-03-01.csv


In [18]:
print("Typ kolumny:", temp_df["temperatura_C"].dtype)
print("\nPodejrzane wartości:")
podejrzane = temp_df[temp_df["temperatura_C"].astype(str).str.contains("C", na=False)]
podejrzane.head()

Typ kolumny: object

Podejrzane wartości:


,timestamp,temperatura_C,uwagi,stacja,data_pliku,wersja,plik
3,2025-03-01 09:00:00,19.9 C,NaN,ST04_gorna,2025-03-01,1,temp_ST04_gorna_2025-03-01.csv
9,2025-03-02 03:00:00,8.22 C,ok,ST03_baluty,2025-03-02,1,temp_ST03_baluty_2025-03-02.csv
12,2025-03-02 12:00:00,9.21 C,NaN,ST03_baluty,2025-03-02,1,temp_ST03_baluty_2025-03-02.csv
18,2025-03-01 06:00:00,17.64 C,ok,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv
19,2025-03-01 09:00:00,1.34 C,NaN,ST01_polesie,2025-03-01,1,temp_ST01_polesie_2025-03-01.csv


## 6. `pandas.Series.str` — czyszczenie kolumn tekstowych

Akcesor `.str` w pandas to **odpowiednik metod stringa, ale zwektoryzowany**.

In [19]:
# Konwersja na string i usunięcie tekstu + spacji
temp_df["temperatura_C"] = (
    temp_df["temperatura_C"]
    .astype(str)
    .str.replace("C", "", regex=False)  # usuń literę C
    .str.strip()                          # usuń białe znaki z brzegów
    .astype(float)                        # teraz można rzutować
)

print("Typ po czyszczeniu:", temp_df["temperatura_C"].dtype)
print(temp_df["temperatura_C"].describe())

Typ po czyszczeniu: float64
count    320.000000
mean      10.938094
std        6.454670
min       -1.750000
25%        5.570000
50%       10.695000
75%       16.310000
max       24.250000
Name: temperatura_C, dtype: float64


### `str.extract` z grupami nazwanymi — najpotężniejsza metoda

Rozbijamy nazwę stacji `ST01_polesie` na kod numeryczny i nazwę dzielnicy:

In [20]:
rozbita = temp_df["stacja"].str.extract(r"ST(?P<kod>\d+)_(?P<dzielnica>\w+)")
rozbita.head()

,kod,dzielnica
0,04,gorna
1,04,gorna
2,04,gorna
3,04,gorna
4,04,gorna


In [21]:
# Dołączamy te kolumny do głównej ramki
temp_df = pd.concat([temp_df, rozbita], axis=1)
temp_df.head()

,timestamp,temperatura_C,uwagi,stacja,data_pliku,wersja,plik,kod,dzielnica
0,2025-03-01 00:00:00,8.45,ok,ST04_gorna,2025-03-01,1,temp_ST04_gorna_2025-03-01.csv,04,gorna
1,2025-03-01 03:00:00,4.85,ok,ST04_gorna,2025-03-01,1,temp_ST04_gorna_2025-03-01.csv,04,gorna
2,2025-03-01 06:00:00,5.91,kalibracja,ST04_gorna,2025-03-01,1,temp_ST04_gorna_2025-03-01.csv,04,gorna
3,2025-03-01 09:00:00,19.90,NaN,ST04_gorna,2025-03-01,1,temp_ST04_gorna_2025-03-01.csv,04,gorna
4,2025-03-01 12:00:00,14.11,NaN,ST04_gorna,2025-03-01,1,temp_ST04_gorna_2025-03-01.csv,04,gorna


### **Ćwiczenie 4**
Napisz funkcję `wczytaj_pm25(root)` analogiczną do `wczytaj_temperature`.
Uwaga:
- separator w plikach to `;` (parametr `sep` w `read_csv`)
- data w nazwie jest w formacie `YYYYMMDD` — sparsuj ją do `pd.Timestamp`
- kolumna `czas` w środku ma format `DD.MM.YYYY HH:MM` — sparsuj do datetime

In [22]:
# Ćwiczenie 4 — rozwiązanie

def wczytaj_pm25(root: Path) -> pd.DataFrame:
    """Wczytuje pliki PM2.5, separator=';', parsuje datę z nazwy i czas w środku."""
    ramki = []
    for f in root.rglob("pm25-*.csv"):
        m = wzorzec_pm25.match(f.name)
        if not m:
            print(f"⚠️  Pomijam: {f.name}")
            continue
        meta = m.groupdict()
        df = pd.read_csv(f, sep=";")
        # Data z nazwy pliku: YYYYMMDD → pd.Timestamp
        data_str = meta["data"]  # np. '20250301'
        df["stacja"] = meta["stacja"]
        df["data_pliku"] = pd.Timestamp(f"{data_str[:4]}-{data_str[4:6]}-{data_str[6:]}")
        # Czas wewnątrz: format DD.MM.YYYY HH:MM
        df["czas"] = pd.to_datetime(df["czas"], format="%d.%m.%Y %H:%M")
        ramki.append(df)
    return pd.concat(ramki, ignore_index=True)

pm_df = wczytaj_pm25(ROOT)
print(f"Kształt PM2.5: {pm_df.shape}")
pm_df.head()

Kształt PM2.5: (160, 5)


,czas,PM2.5_ug_m3,PM10_ug_m3,stacja,data_pliku
0,2025-03-01 00:00:00,44.1,69.1,ST03_baluty,2025-03-01
1,2025-03-01 06:00:00,77.3,96.5,ST03_baluty,2025-03-01
2,2025-03-01 12:00:00,40.9,62.2,ST03_baluty,2025-03-01
3,2025-03-01 18:00:00,56.5,90.0,ST03_baluty,2025-03-01
4,2025-03-08 00:00:00,15.9,20.7,ST02_widzew,2025-03-08


## 7. `pd.concat` z `keys` — MultiIndex na łączeniu

Czasem chcemy zachować informację, **z którego pliku** pochodzi wiersz.

In [23]:
kawalki = {}
for f in list((ROOT / "temperatura").glob("temp_*.csv"))[:3]:
    kawalki[f.stem] = pd.read_csv(f)

duza = pd.concat(kawalki, names=["plik_zrodlowy", "wiersz"])
duza.head(7)

ValueError: No objects to concatenate

## 8. Podsumowanie — pełny pipeline

Złożenie wszystkiego w jedną funkcję, która zwraca **gotową do analizy ramkę** ze wszystkich źródeł.

---

## **Zadanie**

Dopisz funkcję `wczytaj_wilgotnosc(root)` dla plików w `wilgotnosc/`.
Uwagi:
1. Pliki mają **nagłówek z komentarzami** (linie zaczynające się od `#`) — pomiń je `comment="#"`.
2. Data w nazwie jest w formacie `YYYY_MM_DD` (podkreślniki!) — przekonwertuj.
3. Niektóre pliki mają wersję `v2` — w finalnej ramce zostaw tylko **najnowszą wersję** dla każdej (stacja, data).
4. Dołącz wilgotność do `final` z poprzedniego kroku.

**Bonus:** wykryj duplikaty pliku temperatury (`_v2`) i zostaw tylko nowszą wersję — analogicznie.

In [24]:
# Zadanie — rozwiązanie: wczytaj_wilgotnosc

def wczytaj_wilgotnosc(root: Path) -> pd.DataFrame:
    """
    Wczytuje pliki wilgotności (humidity_*.txt).
    - Pomija linie komentarzy (#)
    - Data w nazwie: YYYY_MM_DD
    - Zostaje tylko najnowsza wersja (max numer v) dla każdej (stacja, data, hour)
    """
    ramki = []
    for f in root.rglob("humidity_*.txt"):
        m = wzorzec_hum.match(f.name)
        if not m:
            print(f"⚠️  Pomijam: {f.name}")
            continue
        meta = m.groupdict()
        df = pd.read_csv(f, comment="#")
        df["stacja"] = meta["stacja"]
        df["wersja"] = int(meta["wersja"])
        df["data"] = pd.to_datetime(f"{meta['rok']}-{meta['miesiac']}-{meta['dzien']}")
        ramki.append(df)
    hum = pd.concat(ramki, ignore_index=True)
    # Zostaw tylko najnowszą wersję dla każdej (stacja, data, hour)
    hum = (hum.sort_values("wersja")
              .drop_duplicates(subset=["stacja", "data", "hour"], keep="last")
              .reset_index(drop=True))
    return hum

hum_df = wczytaj_wilgotnosc(ROOT)
print(f"Kształt wilgotności: {hum_df.shape}")
hum_df.head(8)

Kształt wilgotności: (240, 5)


,hour,humidity_pct,stacja,wersja,data
0,0,52.5,ST02_widzew,1,2025-03-06
1,4,47.6,ST02_widzew,1,2025-03-06
2,12,86.4,ST02_widzew,1,2025-03-06
3,8,44.2,ST02_widzew,1,2025-03-06
4,20,82.4,ST02_widzew,1,2025-03-06
5,16,45.6,ST02_widzew,1,2025-03-06
6,20,73.1,ST04_gorna,1,2025-03-09
7,16,56.9,ST04_gorna,1,2025-03-09


In [25]:
# Bonus: deduplikacja plików temperatury (_v2)
# Wczytujemy ponownie z informacją o wersji i usuwamy duplikaty

temp_df2 = wczytaj_temperature(ROOT)
temp_df2["temperatura_C"] = (
    temp_df2["temperatura_C"].astype(str)
    .str.replace("C", "", regex=False).str.strip().astype(float)
)
temp_df2["wersja"] = temp_df2["wersja"].astype(int)
temp_df2["timestamp_dt"] = pd.to_datetime(temp_df2["timestamp"])

# Zostaw tylko nowszą wersję dla każdej (stacja, timestamp)
temp_df2_dedup = (
    temp_df2.sort_values("wersja")
    .drop_duplicates(subset=["stacja", "timestamp_dt"], keep="last")
    .reset_index(drop=True)
)
print(f"Przed dedup: {temp_df2.shape[0]} wierszy")
print(f"Po dedup: {temp_df2_dedup.shape[0]} wierszy")

Przed dedup: 320 wierszy
Po dedup: 320 wierszy


In [26]:
def zbuduj_dataset(root: Path) -> pd.DataFrame:
    """Łączy temperatury, PM2.5 i wilgotność w jedną ramkę po (stacja, czas_h)."""
    # Temperatura - z deduplikacją wersji
    temp = wczytaj_temperature(root)
    temp["temperatura_C"] = (
        temp["temperatura_C"].astype(str)
        .str.replace("C", "", regex=False).str.strip().astype(float)
    )
    temp["wersja"] = temp["wersja"].astype(int)
    temp["timestamp"] = pd.to_datetime(temp["timestamp"])
    temp = (temp.sort_values("wersja")
               .drop_duplicates(subset=["stacja", "timestamp"], keep="last"))
    temp["czas_h"] = temp["timestamp"].dt.floor("6h")

    # PM2.5
    pm = wczytaj_pm25(root)
    pm["czas_h"] = pm["czas"].dt.floor("6h")

    # Agregacja temperatury do okien 6h
    temp_agg = temp.groupby(["stacja", "czas_h"], as_index=False).agg(
        temp_srednia=("temperatura_C", "mean")
    )

    # Połącz PM2.5 + temperatura
    polaczone = pm.merge(temp_agg, on=["stacja", "czas_h"], how="left")

    # Wilgotność
    hum = wczytaj_wilgotnosc(root)
    hum["czas_h"] = hum["data"] + pd.to_timedelta(hum["hour"].astype(int), unit="h")
    hum["czas_h"] = hum["czas_h"].dt.floor("6h")
    hum_agg = hum.groupby(["stacja", "czas_h"], as_index=False).agg(
        wilgotnosc_srednia=("humidity_pct", "mean")
    )
    polaczone = polaczone.merge(hum_agg, on=["stacja", "czas_h"], how="left")

    return polaczone[["stacja", "czas_h", "PM2.5_ug_m3", "PM10_ug_m3", "temp_srednia", "wilgotnosc_srednia"]]

final = zbuduj_dataset(ROOT)
print("Kształt finalnej ramki:", final.shape)
final.head(10)

Kształt finalnej ramki: (160, 6)


,stacja,czas_h,PM2.5_ug_m3,PM10_ug_m3,temp_srednia,wilgotnosc_srednia
0,ST03_baluty,2025-03-01 00:00:00,44.1,69.1,10.105,65.45
1,ST03_baluty,2025-03-01 06:00:00,77.3,96.5,7.525,66.40
2,ST03_baluty,2025-03-01 12:00:00,40.9,62.2,15.450,75.90
3,ST03_baluty,2025-03-01 18:00:00,56.5,90.0,8.555,79.10
4,ST02_widzew,2025-03-08 00:00:00,15.9,20.7,11.880,61.05
5,ST02_widzew,2025-03-08 06:00:00,23.8,31.9,9.875,60.00
6,ST02_widzew,2025-03-08 12:00:00,54.6,66.6,17.730,86.45
7,ST02_widzew,2025-03-08 18:00:00,6.1,11.2,10.780,73.60
8,ST01_polesie,2025-03-02 00:00:00,79.5,114.2,1.025,77.45
9,ST01_polesie,2025-03-02 06:00:00,78.3,135.2,16.020,76.10
